# Fixation mRNN Bottleneck-Dimension Sweep

This notebook trains separate fixation mRNN models with bottleneck dimensions 5, 10, 20, and 30, then evaluates each model in a self-contained section so the training trajectory, weight histograms, and reconstruction metrics can be inspected immediately after each run.

In [ ]:
from pathlib import Path
from dataclasses import replace
from io import BytesIO

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = next(parent for parent in Path.cwd().parents if (parent / "src").exists())

import sys
src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from dal_monte_2022_analysis.ephys.modeling import (
    backproject_replay_outputs_to_firing_rates,
    load_fixation_mrnn_config,
    pc_reconstructed_firing_rate_accuracy,
    reconstruction_accuracy,
    replay_fixation_mrnn_run,
    settings_from_config,
    train_fixation_mrnn_scratch,
    variance_comparison,
)


def display_figure(fig, *, dpi=150):
    buffer = BytesIO()
    fig.savefig(buffer, format="png", dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    buffer.seek(0)
    display(Image(data=buffer.getvalue()))


## 1. Setup

In [ ]:
cfg = load_fixation_mrnn_config(repo_root / "configs" / "ephys_fixation_mrnn.yaml")
base_settings = settings_from_config(cfg)
base_settings.dataset_cfg_path = str(repo_root / "configs" / "dataset.yaml")
base_settings.device = "auto"
base_settings.target_mode = "region_pcs"
base_settings.pca_n_components = 42
base_settings.temporal_basis_count = 0
base_settings.hidden_units = 50
base_settings.activation = "tanh"
base_settings.rec_constrained = False
base_settings.inp_constrained = False
base_settings.spectral_radius = 1.1
base_settings.epochs = 50_000
base_settings.lr = 3e-4
base_settings.loss_fn = "mse"
base_settings.temporal_derivative_loss_scale = 1.0
base_settings.temporal_curvature_loss_scale = 0.5
base_settings.correlation_loss_scale = 0.0
base_settings.variance_loss_scale = 0.0
base_settings.fr_reconstruction_loss_scale = 0.0
base_settings.fr_temporal_derivative_loss_scale = 0.0
base_settings.fr_temporal_curvature_loss_scale = 0.0
base_settings.l1_weight_scale = 0.01
base_settings.l1_rate_scale = 0.0
base_settings.l2_weight_scale = 0.0
base_settings.l2_rate_scale = 0.0
base_settings.gradient_clip_norm = 1.0
base_settings.initialization_mode = "single"
base_settings.overwrite_seed_plan = False
base_settings

## 2. Training, Replay, and Evaluation Helpers

In [ ]:
from collections import OrderedDict


def _as_numpy_tensor(value):
    if hasattr(value, "detach"):
        return value.detach().cpu().numpy()
    return np.asarray(value)


def train_or_load_model(bottleneck_dim, *, overwrite=False, run_mode="train"):
    settings = replace(base_settings, recurrent_bottleneck_dim=int(bottleneck_dim))
    scratch_id = f"bottleneck_dim_{int(bottleneck_dim)}"
    run_root = Path(settings.output_subdir)
    if run_mode == "train":
        result = train_fixation_mrnn_scratch(settings, scratch_id=scratch_id, overwrite=overwrite)
        replay = replay_fixation_mrnn_run(result["run_dir"], device=settings.device)
        return {"settings": settings, "result": result, "replay": replay, "scratch_id": scratch_id}

    from dal_monte_2022_analysis.ephys.modeling import resolve_fixation_mrnn_output_root

    run_dir = resolve_fixation_mrnn_output_root(settings) / "scratch" / scratch_id
    checkpoint = run_dir / "checkpoint_final.pth"
    if not checkpoint.exists():
        raise FileNotFoundError(f"Missing checkpoint for bottleneck {bottleneck_dim}: {checkpoint}")
    replay = replay_fixation_mrnn_run(run_dir, device=settings.device)
    history = pd.read_csv(run_dir / "history.csv")
    return {"settings": settings, "result": {"run_dir": run_dir, "history": history}, "replay": replay, "scratch_id": scratch_id}


def plot_loss_curve(history, *, title):
    fig, ax = plt.subplots(figsize=(8.0, 4.0), dpi=140)
    ax.plot(history["iteration"], history["loss"], color="black", label="total", linewidth=1.6)
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Loss")
    ax.set_title(title)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=8)
    fig.tight_layout()
    return fig


def plot_weight_histograms(model, *, title):
    fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.8), dpi=140)
    within_values = []
    for region in model.region_order:
        block = model._within_region_block(region).detach().cpu().numpy().ravel()
        within_values.append(block)
    within_values = np.concatenate(within_values)
    axes[0].hist(within_values, bins=40, color="tab:blue", alpha=0.8)
    axes[0].set_title("Within-region recurrent weights")
    axes[0].set_xlabel("Weight value")
    axes[0].set_ylabel("Count")
    axes[0].spines[["top", "right"]].set_visible(False)

    inter_values = []
    for (source, target) in model._inter_region_left_params:
        left = model._inter_region_left_params[(source, target)].detach().cpu().numpy().ravel()
        right = model._inter_region_right_params[(source, target)].detach().cpu().numpy().ravel()
        inter_values.append(left)
        inter_values.append(right)
    inter_values = np.concatenate(inter_values) if inter_values else np.array([])
    axes[1].hist(inter_values, bins=40, color="tab:orange", alpha=0.8)
    axes[1].set_title("Inter-region recurrent weights")
    axes[1].set_xlabel("Weight value")
    axes[1].set_ylabel("Count")
    axes[1].spines[["top", "right"]].set_visible(False)

    fig.suptitle(title, y=1.02)
    fig.tight_layout()
    return fig


def summarize_reconstruction_metrics(replay):
    pc_metrics = reconstruction_accuracy(replay)
    pc_metrics.insert(0, "metric_family", "pc")
    fr_metrics = pc_reconstructed_firing_rate_accuracy(replay)
    fr_metrics.insert(0, "metric_family", "fr")
    return pc_metrics, fr_metrics


def show_metrics_table(pc_metrics, fr_metrics):
    display(pd.concat([pc_metrics, fr_metrics], ignore_index=True).round(4))


## 3. Model: Bottleneck 5

In [ ]:
bottleneck_dim = 5
fit = train_or_load_model(bottleneck_dim, overwrite=True, run_mode="train")
replay = fit["replay"]
model = replay["model"]

loss_fig = plot_loss_curve(fit["result"]["history"], title=f"Bottleneck {bottleneck_dim}: loss trajectory")
display_figure(loss_fig)

weight_fig = plot_weight_histograms(model, title=f"Bottleneck {bottleneck_dim}: recurrent weight histograms")
display_figure(weight_fig)

pc_metrics, fr_metrics = summarize_reconstruction_metrics(replay)
show_metrics_table(pc_metrics, fr_metrics)

# Display a compact region-wise summary for quick comparison.
summary = pd.concat([pc_metrics, fr_metrics], ignore_index=True).groupby(["metric_family", "region"], sort=False)[["r2", "correlation", "mse", "mae"]].mean(numeric_only=True).reset_index()
display(summary.round(4))

## 4. Model: Bottleneck 10

In [ ]:
bottleneck_dim = 10
fit = train_or_load_model(bottleneck_dim, overwrite=True, run_mode="train")
replay = fit["replay"]
model = replay["model"]

loss_fig = plot_loss_curve(fit["result"]["history"], title=f"Bottleneck {bottleneck_dim}: loss trajectory")
display_figure(loss_fig)

weight_fig = plot_weight_histograms(model, title=f"Bottleneck {bottleneck_dim}: recurrent weight histograms")
display_figure(weight_fig)

pc_metrics, fr_metrics = summarize_reconstruction_metrics(replay)
show_metrics_table(pc_metrics, fr_metrics)

summary = pd.concat([pc_metrics, fr_metrics], ignore_index=True).groupby(["metric_family", "region"], sort=False)[["r2", "correlation", "mse", "mae"]].mean(numeric_only=True).reset_index()
display(summary.round(4))

## 5. Model: Bottleneck 20

In [ ]:
bottleneck_dim = 20
fit = train_or_load_model(bottleneck_dim, overwrite=True, run_mode="train")
replay = fit["replay"]
model = replay["model"]

loss_fig = plot_loss_curve(fit["result"]["history"], title=f"Bottleneck {bottleneck_dim}: loss trajectory")
display_figure(loss_fig)

weight_fig = plot_weight_histograms(model, title=f"Bottleneck {bottleneck_dim}: recurrent weight histograms")
display_figure(weight_fig)

pc_metrics, fr_metrics = summarize_reconstruction_metrics(replay)
show_metrics_table(pc_metrics, fr_metrics)

summary = pd.concat([pc_metrics, fr_metrics], ignore_index=True).groupby(["metric_family", "region"], sort=False)[["r2", "correlation", "mse", "mae"]].mean(numeric_only=True).reset_index()
display(summary.round(4))

## 6. Model: Bottleneck 30

In [ ]:
bottleneck_dim = 30
fit = train_or_load_model(bottleneck_dim, overwrite=True, run_mode="train")
replay = fit["replay"]
model = replay["model"]

loss_fig = plot_loss_curve(fit["result"]["history"], title=f"Bottleneck {bottleneck_dim}: loss trajectory")
display_figure(loss_fig)

weight_fig = plot_weight_histograms(model, title=f"Bottleneck {bottleneck_dim}: recurrent weight histograms")
display_figure(weight_fig)

pc_metrics, fr_metrics = summarize_reconstruction_metrics(replay)
show_metrics_table(pc_metrics, fr_metrics)

summary = pd.concat([pc_metrics, fr_metrics], ignore_index=True).groupby(["metric_family", "region"], sort=False)[["r2", "correlation", "mse", "mae"]].mean(numeric_only=True).reset_index()
display(summary.round(4))

## 7. Compare Reconstruction Metrics Across Models

In [ ]:
comparison_rows = []
for bottleneck_dim in [5, 10, 20, 30]:
    fit = train_or_load_model(bottleneck_dim, overwrite=False, run_mode="train")
    replay = fit["replay"]
    pc_metrics, fr_metrics = summarize_reconstruction_metrics(replay)
    for _, row in pc_metrics.groupby(["region"], sort=False).mean(numeric_only=True).reset_index().iterrows():
        comparison_rows.append({"bottleneck_dim": bottleneck_dim, "metric_family": "pc", "region": row["region"], "r2": row["r2"], "correlation": row["correlation"], "mse": row["mse"], "mae": row["mae"]})
    for _, row in fr_metrics.groupby(["region"], sort=False).mean(numeric_only=True).reset_index().iterrows():
        comparison_rows.append({"bottleneck_dim": bottleneck_dim, "metric_family": "fr", "region": row["region"], "r2": row["r2"], "correlation": row["correlation"], "mse": row["mse"], "mae": row["mae"]})

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df.round(4))

if not comparison_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(11.0, 3.8), dpi=140, sharey=True)
    for metric_family, ax in zip(["pc", "fr"], axes):
        sub = comparison_df[comparison_df["metric_family"] == metric_family]
        for region in sorted(sub["region"].unique()):
            region_sub = sub[sub["region"] == region]
            ax.plot(region_sub["bottleneck_dim"], region_sub["r2"], marker="o", label=region)
        ax.set_xlabel("Bottleneck dimension")
        ax.set_ylabel("Mean R2")
        ax.set_title(f"{metric_family.upper()} reconstruction R2")
        ax.spines[["top", "right"]].set_visible(False)
        ax.legend(frameon=False, fontsize=7)
    fig.tight_layout()
    display_figure(fig)